# Entropy Analysis for Experiment Results
This notebook analyzes entropy metrics from experiment rollouts and visualizes patterns between correct and incorrect solutions.

## 1. Imports and Setup

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Experiment Data

In [ ]:
# Load experiment results
with open('experiments/experiment_20251027_231010/experiment.json', 'r') as f:
    experiment = json.load(f)

# Find problem_401 key
problem_key = next(k for k in experiment['results'] if experiment['results'][k]['dataset_index'] == 401)
num_rollouts = len(experiment['results'][problem_key]['rollouts'])

print(f"Loaded experiment with {num_rollouts} rollouts for problem 401")

## 3. Define Metric Functions

In [ ]:
def compute_per_step_metrics(entropy, window=10):
    """Compute per-step metrics for entropy sequence."""
    n = len(entropy)
    
    # Cumulative sum
    cumsum = np.cumsum(entropy)
    
    # Rolling volatility (std)
    volatility = np.full(n, np.nan)
    for i in range(window, n):
        volatility[i] = np.std(entropy[i-window:i])
    
    # Rolling slope
    slope = np.full(n, np.nan)
    for i in range(window, n):
        slope[i] = np.polyfit(range(window), entropy[i-window:i], 1)[0]
    
    return {
        'cumsum': cumsum,
        'volatility': volatility,
        'slope': slope
    }

In [ ]:
def compute_global_metrics(entropy, window=10):
    """Compute global metrics for entropy sequence."""
    # Basic statistics
    mean_val = np.mean(entropy)
    median_val = np.median(entropy)
    std_val = np.std(entropy)
    
    # Compute deltas
    deltas = np.diff(entropy)
    abs_deltas = np.abs(deltas)
    
    # Spikes: tokens where both incoming and outgoing deltas are in top 5%
    threshold_95 = np.percentile(abs_deltas, 95)
    num_spikes = 0
    for i in range(1, len(entropy) - 1):
        delta_prev = abs(entropy[i] - entropy[i-1])
        delta_next = abs(entropy[i+1] - entropy[i])
        if delta_prev >= threshold_95 and delta_next >= threshold_95:
            num_spikes += 1
    
    # Phase shifts: detect significant changes in rolling mean/std
    rolling_mean = np.full(len(entropy), np.nan)
    rolling_std = np.full(len(entropy), np.nan)
    for i in range(window, len(entropy)):
        rolling_mean[i] = np.mean(entropy[i-window:i])
        rolling_std[i] = np.std(entropy[i-window:i])
    
    # Detect phase shifts: large changes in rolling statistics
    mean_changes = np.abs(np.diff(rolling_mean[window:]))
    std_changes = np.abs(np.diff(rolling_std[window:]))
    
    mean_change_threshold = np.percentile(mean_changes[~np.isnan(mean_changes)], 90) if len(mean_changes) > 0 else 0
    std_change_threshold = np.percentile(std_changes[~np.isnan(std_changes)], 90) if len(std_changes) > 0 else 0
    
    phase_shifts = 0
    for i in range(len(mean_changes)):
        if (not np.isnan(mean_changes[i]) and mean_changes[i] >= mean_change_threshold) or \
           (not np.isnan(std_changes[i]) and std_changes[i] >= std_change_threshold):
            phase_shifts += 1
    
    return {
        'mean': mean_val,
        'median': median_val,
        'std': std_val,
        'num_spikes': num_spikes,
        'phase_shifts': phase_shifts
    }

## 4. Process All Rollouts

In [ ]:
# Load all rollouts and compute metrics
rollout_data = []
for rollout_num in range(num_rollouts):
    with open(f'experiments/experiment_20251027_231010/problem_401/rollout_{rollout_num}/graph_structure.json', 'r') as f:
        data = json.load(f)
    
    entropy = np.array([item['metric_value'] for item in data['final_sequence']])
    is_correct = experiment['results'][problem_key]['rollouts'][rollout_num]['is_correct']
    
    # Compute metrics
    per_step = compute_per_step_metrics(entropy)
    global_metrics = compute_global_metrics(entropy)
    
    rollout_data.append({
        'rollout_num': rollout_num,
        'entropy': entropy,
        'is_correct': is_correct,
        'per_step': per_step,
        'global': global_metrics
    })

print("Computed all metrics for all rollouts")
print(f"Total rollouts: {len(rollout_data)}")
print(f"Correct: {sum(1 for r in rollout_data if r['is_correct'])}")
print(f"Wrong: {sum(1 for r in rollout_data if not r['is_correct'])}")

## 5. Visualization 1: Original Entropy Plot

In [ ]:
plt.figure(figsize=(14, 6))
for rollout in rollout_data:
    color = 'green' if rollout['is_correct'] else 'red'
    label = f"Rollout {rollout['rollout_num']} ({'CORRECT' if rollout['is_correct'] else 'WRONG'})"
    plt.plot(rollout['entropy'], linewidth=1.5, color=color, alpha=0.7, label=label)

plt.xlabel('Token Position')
plt.ylabel('Entropy')
plt.title('Entropy Values for All Rollouts (Problem 401)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visualization 2: Per-step Metrics

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Cumulative Sum
for rollout in rollout_data:
    color = 'green' if rollout['is_correct'] else 'red'
    label = f"Rollout {rollout['rollout_num']}"
    axes[0].plot(rollout['per_step']['cumsum'], linewidth=1.5, color=color, alpha=0.7, label=label)
axes[0].set_xlabel('Token Position')
axes[0].set_ylabel('Cumulative Entropy')
axes[0].set_title('Cumulative Sum of Entropy')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Volatility
for rollout in rollout_data:
    color = 'green' if rollout['is_correct'] else 'red'
    label = f"Rollout {rollout['rollout_num']}"
    axes[1].plot(rollout['per_step']['volatility'], linewidth=1.5, color=color, alpha=0.7, label=label)
axes[1].set_xlabel('Token Position')
axes[1].set_ylabel('Rolling Std (window=10)')
axes[1].set_title('Entropy Volatility')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Slope
for rollout in rollout_data:
    color = 'green' if rollout['is_correct'] else 'red'
    label = f"Rollout {rollout['rollout_num']}"
    axes[2].plot(rollout['per_step']['slope'], linewidth=1.5, color=color, alpha=0.7, label=label)
axes[2].set_xlabel('Token Position')
axes[2].set_ylabel('Rolling Slope (window=10)')
axes[2].set_title('Entropy Slope (Trend)')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Visualization 3: Global Metrics - Box Plots

In [ ]:
# Separate data by correctness
correct_data = [r for r in rollout_data if r['is_correct']]
wrong_data = [r for r in rollout_data if not r['is_correct']]

metrics_to_plot = ['mean', 'median', 'std', 'num_spikes', 'phase_shifts']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 5))

for idx, metric in enumerate(metrics_to_plot):
    correct_values = [r['global'][metric] for r in correct_data]
    wrong_values = [r['global'][metric] for r in wrong_data]
    
    box_data = [correct_values, wrong_values]
    bp = axes[idx].boxplot(box_data, labels=['Correct', 'Wrong'], patch_artist=True)
    
    # Color the boxes
    bp['boxes'][0].set_facecolor('lightgreen')
    bp['boxes'][1].set_facecolor('lightcoral')
    
    axes[idx].set_title(f'{metric.replace("_", " ").title()}')
    axes[idx].set_ylabel('Value')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.suptitle('Global Metrics Comparison: Correct vs Wrong Rollouts (Box Plots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Visualization 4: Global Metrics - Bar Charts

In [ ]:
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 5))

x_pos = np.arange(num_rollouts)
for idx, metric in enumerate(metrics_to_plot):
    values = [r['global'][metric] for r in rollout_data]
    colors = ['green' if r['is_correct'] else 'red' for r in rollout_data]
    
    axes[idx].bar(x_pos, values, color=colors, alpha=0.7)
    axes[idx].set_title(f'{metric.replace("_", " ").title()}')
    axes[idx].set_xlabel('Rollout')
    axes[idx].set_ylabel('Value')
    axes[idx].set_xticks(x_pos)
    axes[idx].set_xticklabels([f'R{i}' for i in range(num_rollouts)])
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.suptitle('Global Metrics for Each Rollout (Bar Charts)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Print Summary

In [ ]:
print("\n" + "="*80)
print("GLOBAL METRICS SUMMARY")
print("="*80)
for rollout in rollout_data:
    status = "CORRECT" if rollout['is_correct'] else "WRONG"
    print(f"\nRollout {rollout['rollout_num']} ({status}):")
    for metric, value in rollout['global'].items():
        print(f"  {metric}: {value:.4f}" if isinstance(value, float) else f"  {metric}: {value}")